# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{getattr(metadata, 'name', 'N/A')}: {getattr(metadata, 'description', 'No description found.')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets, their `@id`s and fields
print('Available record sets:')
record_sets = list(dataset.metadata.record_sets)
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}, name: {rs.name}")
    if hasattr(rs, 'fields'):
        for field in rs.fields:
            print(f"  Field @id: {field.id}, name: {field.name}, dataType: {field.data_type}")
    elif hasattr(rs, 'columns'):
        for col in rs.columns:
            print(f"  Column @id: {col.id}, name: {col.name}, dataType: {col.data_type}")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use record set and field `@id`s from the overview.

In [ ]:
# Example: extract all data from top-level record sets
import collections

# Collect all record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

dataframes = collections.OrderedDict()
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded record set: {record_set_id}")
    print("Columns:", df.columns.tolist())

# Show example rows from the first record set
if record_set_ids:
    first_set = record_set_ids[0]
    display(dataframes[first_set].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For demonstration, we select a numeric field from the first record set
import numpy as np

if record_set_ids:
    df = dataframes[first_set]
    # Try to find a numeric field (float or int) automatically
    num_fields = []
    for rs in dataset.metadata.record_sets:
        if rs.id == first_set:
            for f in getattr(rs, 'fields', getattr(rs, 'columns', [])):
                if f.data_type in ['schema:Float', 'schema:Integer', 'Float', 'Integer']:
                    num_fields.append(f.id)
    if num_fields:
        numeric_field = num_fields[0]
        print(f"Using numeric field: {numeric_field}")
        
        # Drop missing values for this numeric field
        filtered_df = df.dropna(subset=[numeric_field])
        # Filter for values greater than threshold
        threshold = np.percentile(filtered_df[numeric_field], 80)
        filtered_df = filtered_df[filtered_df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df[[numeric_field]].head())
        
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        
        # Try grouping by another field if available
        group_fields = [f.id for f in getattr(rs, 'fields', getattr(rs, 'columns', [])) if f.data_type == 'schema:Text']
        if group_fields:
            group_field = group_fields[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field} (mean {numeric_field}):")
            print(grouped_df.head())
    else:
        print("No numeric field found for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Visualize the distribution of the numeric field in the filtered DataFrame
if record_set_ids and num_fields:
    nf = numeric_field
    filtered_df[nf].hist(bins=20)
    plt.xlabel(nf)
    plt.ylabel('Frequency')
    plt.title(f'Distribution of {nf} (filtered)')
    plt.show()
    
    # If there is a group_field, visualize mean per group
    if group_fields:
        grouped_df.plot(kind='bar', x=group_field, y=numeric_field, legend=False)
        plt.ylabel(f'Mean {nf}')
        plt.title(f'Mean {nf} by {group_field}')
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.


*This notebook demonstrates loading and exploring the FAIR² dataset “Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells” via the `mlcroissant` library. We identified available record sets, programmatically discovered fields (including numeric fields useful for analysis), visualized sample distributions, and outlined a reproducible pattern for Croissant-compatible datasets. For further analyses, continue with domain-specific filtering and feature engineering based on record set field meanings.*